In [293]:
import numpy as np
import pandas as pd
import scipy.io
import matplotlib.pyplot as plt
import math

In [423]:
sol = scipy.io.loadmat('./Solutions/alpha_1_526.mat')

In [424]:
sol_struct = sol['sol']
maxCl = sol_struct[0,0]

In [425]:
#print(maxCl['belongTo'])
maxCl['belongTo'] = np.squeeze(maxCl['belongTo'])
print(maxCl['belongTo'])

[7 3 6 6 6 6 5 6 6 6 6 1 6 6 6 6 6 6 6 6 6 6 6 6 2 6 6 6 6 6 6 6 7 3 6 6 6
 6 6 6 7 6 6 7 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 5 7 6 6 6 6 6 6 6 7 6 5 6
 6 6 6 6 6 6 5 2 7 6 6 6 6 6 6 6 6 5 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6
 6 6 6 6 6 6 6 6 2 7 6 3 6 6 6 6 7 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 5 6 6 6 6
 4 6 6 6 2 6 6 6 6 6 6 6 6 6 6 6 4 6 6 6 6 6 6 6 5 6 6 6 6 6 6 5 6 6 6 6 6
 6 7 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 5 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6
 6 5 6 7 6 6 6 6 6 6 6 6 6 6 3 6 6 7 6 6 6 6 6 6 6 6 6 3 6 6 6 5 6 6 6 6 6
 6 7 6 6 6 7 6 6 3 6 6 6 6 6 6 6 6 6 6 6 6 7 3 6 6 6 6 6 6 6 7 6 6 6 5 6 6
 5 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 7 6 3 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6
 6 5 6 5 6 6 6 6 6 6 7 6 3 6 6 6 6 6 7 6 6 6 6 6 6 6 5 5 6 6 6 6 6 7 6 6 5
 6 5 6 6 6 6 6 4 6 7 6 6 6 6 6 6 6 6 6 7 6 6 6 6 5 6 6 6 6 6 6 6 6 6 6 7 6
 6 6 7 6 6 7 6 7 6 6 3 6 6 6 6 6 3 6 6 6 6 6 6 6 6 6 6 6 5 6 6 6 6 6 4 3 5
 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 7 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6
 6 6 6 6 7 6 6 6 6 6 6 6 

In [426]:
print(maxCl['weight'])

[[-0.15760324 -0.63584513  0.87923972  0.47478826 -0.37282979  0.52669062
  -0.23234714]
 [ 0.2037592  -0.72891915  0.51117923  0.58766681  0.10662716  0.57368234
   0.55254617]
 [ 0.744893   -0.45445677  0.13710364  0.54203572  0.58409612  0.64785518
   0.85929259]
 [-0.6656341   0.27128701 -0.15397104 -0.54119288  0.59662817 -0.23430787
  -0.29725374]
 [-0.91325422  0.61237584  0.20420257 -0.56384258  0.72651708 -0.29278576
   0.0961269 ]
 [-0.76562689  0.33575465  0.32204391 -0.64982067  0.59592441 -0.33896347
   0.12896507]]


In [427]:
reward_func = maxCl['weight']
reward_reshape = reward_func.T
print(reward_reshape)

[[-0.15760324  0.2037592   0.744893   -0.6656341  -0.91325422 -0.76562689]
 [-0.63584513 -0.72891915 -0.45445677  0.27128701  0.61237584  0.33575465]
 [ 0.87923972  0.51117923  0.13710364 -0.15397104  0.20420257  0.32204391]
 [ 0.47478826  0.58766681  0.54203572 -0.54119288 -0.56384258 -0.64982067]
 [-0.37282979  0.10662716  0.58409612  0.59662817  0.72651708  0.59592441]
 [ 0.52669062  0.57368234  0.64785518 -0.23430787 -0.29278576 -0.33896347]
 [-0.23234714  0.55254617  0.85929259 -0.29725374  0.0961269   0.12896507]]


In [431]:
policies = maxCl['policy']
policies = policies.T
np.shape(policies)

(7, 750)

In [387]:
mimic = pd.read_csv("./dataSets/Sepsis_data_process.csv")
demo_states = np.load("./dataSets/demonstrations_states_19.npy")
demo_actions = np.load("./dataSets/demonstrations_actions_19.npy")


In [388]:
tupleToNumericStates = {}
numericStatesToTuple = {}
count = 0
for i in range(25):
    for j in range(30):
        tupleToNumericStates[(i,j)] = count
        numericStatesToTuple[count] = (i,j)
        count += 1


In [389]:
# recreate the original mimic table
def recreateOriginalTable(demonstrations, learnt_policy):
    icuids = list(map(lambda x: x[20], demonstrations))
    
    mimic_new = mimic[mimic['icustayid'].isin(icuids)]

    # map learnt actions to states
    mapActions(mimic_new, learnt_policy)
    
    return mimic_new, len(icuids)


In [390]:
def mapActions(mimic_table, learnt_policy):
    actions = []
    for i in range(len(mimic_table)):
        curr_state = mimic_table.iloc[i]['discrete_states']
        action_for_state = learnt_policy[int(curr_state)]
        action_for_state -= 1
        actions.append(action_for_state)
    mimic_table['piL'] = actions

In [307]:
table = recreateOriginalTable(demo_states)

TypeError: recreateOriginalTable() missing 1 required positional argument: 'learnt_policy'

In [391]:
class ClusterObject:
    def __init__(self, index, size, table, reward_function):
        self.no_trajs = size
        self.clusterId = index
        self.table = table
        self.reward = reward_function
        self.calculateRewards()
        self.policyMatch = self.calculatePolicyMatching()
        self.statistics_sofa, self.statistics_AL = self.statistics()
        
    def calculatePolicyMatching(self):
        tot = 0 
        matches = 0
        for i in range(len(self.table)):
            piE = self.table.iloc[i]['actions']
            piL = self.table.iloc[i]['piL']
            if piL == piE:
                matches += 1
            tot += 1
        return (matches / tot) * 100
    
    
    def calculateRewards(self):
        rewards = []
        print(self.reward.T)
        for i in range(len(self.table) - 1):
            feature_vector = [0 for i in range(6)]
            cur_sofa = self.table.iloc[i]['SOFA']
            next_sofa = self.table.iloc[i+1]['SOFA']
            
            cur_lact = self.table.iloc[i]['Arterial_lactate']
            next_lact = self.table.iloc[i+1]['Arterial_lactate']
            
            cur_lact = math.floor(cur_lact)
            next_lact = math.floor(next_lact)
            
            if cur_sofa > next_sofa:
                feature_vector[2] = 1
            if cur_sofa == next_sofa:
                feature_vector[1] = 1
            if cur_sofa < next_sofa:
                feature_vector[0] = 1
                
            if cur_lact > next_lact:
                feature_vector[5] = 1
            if cur_lact == next_lact:
                feature_vector[4] = 1
            if cur_lact < next_lact:
                feature_vector[3] = 1
            print(feature_vector)
            
            reward = np.dot(feature_vector, self.reward.T)
            print(reward)
            rewards.append(reward)
        rewards.append(-1)
        self.table['reward'] = rewards
        
    def statistics(self):
        #. count number of times SOFA inc, maintain, dec
        statistics_sofa = [0 for i in range(3)]
        statistics_AL = [0 for i in range(3)]
        for i in range(len(self.table) - 1):
            curr_sofa = self.table.iloc[i]['SOFA']
            next_sofa = self.table.iloc[i+1]['SOFA']
            
            curr_lactate = self.table.iloc[i]['Arterial_lactate']
            next_lactate = self.table.iloc[i+1]['Arterial_lactate']
            
            curr_lactate = math.floor(curr_lactate)
            next_lactate = math.floor(next_lactate)
            
            if curr_sofa > next_sofa:
                statistics_sofa[2] += 1
            if curr_sofa == next_sofa:
                statistics_sofa[1] += 1
            if curr_sofa < next_sofa:
                statistics_sofa[0] += 1
                
            if curr_lactate > next_lactate:
                statistics_AL[2] += 1
            if curr_lactate == next_lactate:
                statistics_AL[1] += 1
            if curr_lactate < next_lactate:
                statistics_AL[0] += 1

        return [x / sum(statistics_sofa) for x in statistics_sofa], [x / sum(statistics_AL) for x in statistics_AL]
    
    def plot(self):
        pass
                
        

In [392]:
def createClusterDictionary(mimic, maxCl, demonstrations, rewards, piL):
    dct = {}
    belongTo = maxCl['belongTo']
    
    for i in range(1, int(max(belongTo)) + 1):
        indices = np.where(belongTo == i) # indexs of cluster assignemnt = i 
        trajectory_clusters = demonstrations[indices]
        
        # 750x1 vector
        cluster_policy = piL[i - 1]
        
        # 6x1 vector
        cluster_reward = rewards[i - 1]
        
        cluster_table, size = recreateOriginalTable(trajectory_clusters, cluster_policy)

        # append policies
        
        #print(len(cluster_table))
        dct[i] = ClusterObject(i, size, cluster_table, cluster_reward)
    
    return dct

In [393]:
clusterDct = createClusterDictionary(table, maxCl, demo_states, reward_reshape, policies)

/Users/ongyichong/anaconda3/lib/python3.6/site-packages/ipykernel_launcher.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/Users/ongyichong/anaconda3/lib/python3.6/site-packages/ipykernel_launcher.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


[-0.15760324  0.2037592   0.744893   -0.6656341  -0.91325422 -0.76562689]
[1, 0, 0, 1, 0, 0]
-0.8232373423236197
[0, 0, 1, 0, 1, 0]
-0.16836121969135132
[1, 0, 0, 0, 1, 0]
-1.070857468812514
[1, 0, 0, 0, 1, 0]
-1.070857468812514
[0, 0, 1, 0, 1, 0]
-0.16836121969135132
[0, 1, 0, 0, 1, 0]
-0.7094950200501631
[1, 0, 0, 0, 1, 0]
-1.070857468812514
[0, 0, 1, 0, 0, 1]
-0.020733883709019185
[1, 0, 0, 0, 1, 0]
-1.070857468812514
[1, 0, 0, 1, 0, 0]
-0.8232373423236197
[0, 0, 1, 0, 1, 0]
-0.16836121969135132
[1, 0, 0, 0, 1, 0]
-1.070857468812514
[0, 0, 1, 0, 1, 0]
-0.16836121969135132
[0, 1, 0, 0, 1, 0]
-0.7094950200501631
[1, 0, 0, 0, 1, 0]
-1.070857468812514
[0, 1, 0, 0, 1, 0]
-0.7094950200501631
[0, 0, 1, 0, 1, 0]
-0.16836121969135132
[0, 1, 0, 0, 1, 0]
-0.7094950200501631
[0, 0, 1, 1, 0, 0]
0.0792589067975431
[0, 0, 1, 0, 0, 1]
-0.020733883709019185
[0, 0, 1, 0, 1, 0]
-0.16836121969135132
[0, 1, 0, 1, 0, 0]
-0.46187489356126865
[1, 0, 0, 0, 0, 1]
-0.923230132830182
[0, 1, 0, 0, 1, 0]
-0.7094

[0, 0, 1, 0, 1, 0]
0.34130621582216936
[0, 0, 1, 0, 0, 1]
0.4591475524176858
[0, 1, 0, 0, 1, 0]
0.7153818002218011
[0, 1, 0, 1, 0, 0]
0.3572081940037779
[0, 1, 0, 0, 1, 0]
0.7153818002218011
[0, 1, 0, 0, 0, 1]
0.8332231368173175
[0, 1, 0, 0, 1, 0]
0.7153818002218011
[0, 1, 0, 0, 1, 0]
0.7153818002218011
[0, 1, 0, 1, 0, 0]
0.3572081940037779
[0, 1, 0, 0, 1, 0]
0.7153818002218011
[0, 1, 0, 0, 1, 0]
0.7153818002218011
[0, 1, 0, 0, 1, 0]
0.7153818002218011
[0, 1, 0, 0, 0, 1]
0.8332231368173175
[1, 0, 0, 0, 1, 0]
1.0834422865857
[0, 0, 1, 0, 0, 1]
0.4591475524176858
[1, 0, 0, 0, 1, 0]
1.0834422865857
[0, 1, 0, 1, 0, 0]
0.3572081940037779
[0, 0, 1, 0, 1, 0]
0.34130621582216936
[0, 0, 1, 0, 0, 1]
0.4591475524176858
[0, 1, 0, 0, 1, 0]
0.7153818002218011
[0, 1, 0, 0, 1, 0]
0.7153818002218011
[1, 0, 0, 0, 1, 0]
1.0834422865857
[0, 0, 1, 0, 1, 0]
0.34130621582216936
[0, 1, 0, 0, 1, 0]
0.7153818002218011
[1, 0, 0, 0, 1, 0]
1.0834422865857
[0, 0, 1, 1, 0, 0]
-0.016867390395853832
[0, 0, 1, 0, 0, 1]

[1, 0, 0, 0, 1, 0]
0.35368729085431316
[0, 0, 1, 0, 0, 1]
1.1800205338894
[1, 0, 0, 0, 1, 0]
0.35368729085431316
[0, 0, 1, 0, 1, 0]
1.3106132061145839
[0, 1, 0, 1, 0, 0]
0.7032553260353098
[1, 0, 0, 0, 0, 1]
0.22309461862912938
[0, 0, 1, 1, 0, 0]
1.180724291155201
[1, 0, 0, 0, 1, 0]
0.35368729085431316
[0, 0, 1, 0, 0, 1]
1.1800205338894
[0, 0, 1, 0, 1, 0]
1.3106132061145839
[1, 0, 0, 1, 0, 0]
0.22379837589493023
[0, 0, 1, 0, 1, 0]
1.3106132061145839
[0, 0, 1, 0, 0, 1]
1.1800205338894
[0, 0, 1, 0, 0, 1]
1.1800205338894
[0, 1, 0, 0, 1, 0]
0.8331442409946928
[0, 0, 1, 1, 0, 0]
1.180724291155201
[1, 0, 0, 1, 0, 0]
0.22379837589493023
[1, 0, 0, 0, 0, 1]
0.22309461862912938
[0, 0, 1, 0, 1, 0]
1.3106132061145839
[1, 0, 0, 0, 1, 0]
0.35368729085431316
[0, 0, 1, 0, 1, 0]
1.3106132061145839
[0, 0, 1, 1, 0, 0]
1.180724291155201
[0, 0, 1, 0, 1, 0]
1.3106132061145839
[1, 0, 0, 0, 0, 1]
0.22309461862912938
[1, 0, 0, 0, 1, 0]
0.35368729085431316
[0, 0, 1, 0, 1, 0]
1.3106132061145839
[0, 0, 1, 0, 1, 0

[ 0.52669062  0.57368234  0.64785518 -0.23430787 -0.29278576 -0.33896347]
[1, 0, 0, 0, 0, 1]
0.18772714961801595
[0, 1, 0, 1, 0, 0]
0.3393744725436567
[0, 0, 1, 0, 1, 0]
0.35506941737276865
[0, 0, 1, 0, 1, 0]
0.35506941737276865
[0, 1, 0, 1, 0, 0]
0.3393744725436567
[0, 1, 0, 1, 0, 0]
0.3393744725436567
[0, 1, 0, 0, 0, 1]
0.2347188696269331
[1, 0, 0, 0, 1, 0]
0.23390486377965014
[0, 1, 0, 0, 1, 0]
0.2808965837885673
[0, 0, 1, 1, 0, 0]
0.41354730612785806
[0, 1, 0, 0, 1, 0]
0.2808965837885673
[0, 1, 0, 0, 1, 0]
0.2808965837885673
[0, 1, 0, 0, 1, 0]
0.2808965837885673
[0, 0, 1, 0, 1, 0]
0.35506941737276865
[1, 0, 0, 0, 0, 1]
0.18772714961801595
[1, 0, 0, 0, 1, 0]
0.23390486377965014
[1, 0, 0, 0, 1, 0]
0.23390486377965014
[0, 0, 1, 0, 1, 0]
0.35506941737276865
[1, 0, 0, 1, 0, 0]
0.29238275253473955
[1, 0, 0, 0, 1, 0]
0.23390486377965014
[0, 1, 0, 0, 0, 1]
0.2347188696269331
[0, 0, 1, 0, 1, 0]
0.35506941737276865
[0, 0, 1, 0, 1, 0]
0.35506941737276865
[1, 0, 0, 1, 0, 0]
0.29238275253473955

...

[0, 0, 1, 0, 1, 0]
0.35506941737276865
[0, 1, 0, 0, 1, 0]
0.2808965837885673
[1, 0, 0, 0, 1, 0]
0.23390486377965014
[0, 0, 1, 0, 1, 0]
0.35506941737276865
[1, 0, 0, 0, 0, 1]
0.18772714961801595
[0, 0, 1, 0, 1, 0]
0.35506941737276865
[0, 0, 1, 1, 0, 0]
0.41354730612785806
[1, 0, 0, 0, 0, 1]
0.18772714961801595
[0, 1, 0, 0, 1, 0]
0.2808965837885673
[0, 0, 1, 1, 0, 0]
0.41354730612785806
[1, 0, 0, 0, 1, 0]
0.23390486377965014
[0, 1, 0, 0, 1, 0]
0.2808965837885673
[1, 0, 0, 0, 1, 0]
0.23390486377965014
[1, 0, 0, 0, 1, 0]
0.23390486377965014
[0, 0, 1, 0, 0, 1]
0.30889170321113446
[1, 0, 0, 1, 0, 0]
0.29238275253473955
[0, 0, 1, 0, 1, 0]
0.35506941737276865
[1, 0, 0, 0, 0, 1]
0.18772714961801595
[1, 0, 0, 0, 1, 0]
0.23390486377965014
[0, 0, 1, 1, 0, 0]
0.41354730612785806
[0, 0, 1, 0, 1, 0]
0.35506941737276865
[1, 0, 0, 0, 1, 0]
0.23390486377965014
[0, 1, 0, 0, 0, 1]
0.2347188696269331
[0, 0, 1, 1, 0, 0]
0.41354730612785806
[0, 0, 1, 0, 1, 0]
0.35506941737276865
[0, 1, 0, 0, 1, 0]
0.28089658

[1, 0, 0, 0, 0, 1]
0.18772714961801595
[0, 0, 1, 1, 0, 0]
0.41354730612785806
[1, 0, 0, 0, 0, 1]
0.18772714961801595
[0, 1, 0, 0, 1, 0]
0.2808965837885673
[0, 1, 0, 0, 1, 0]
0.2808965837885673
[0, 1, 0, 0, 1, 0]
0.2808965837885673
[1, 0, 0, 0, 0, 1]
0.18772714961801595
[1, 0, 0, 1, 0, 0]
0.29238275253473955
[0, 1, 0, 0, 1, 0]
0.2808965837885673
[0, 0, 1, 0, 1, 0]
0.35506941737276865
[1, 0, 0, 0, 1, 0]
0.23390486377965014
[0, 1, 0, 0, 1, 0]
0.2808965837885673
[1, 0, 0, 0, 0, 1]
0.18772714961801595
[0, 0, 1, 1, 0, 0]
0.41354730612785806
[1, 0, 0, 0, 0, 1]
0.18772714961801595
[0, 1, 0, 1, 0, 0]
0.3393744725436567
[0, 0, 1, 0, 0, 1]
0.30889170321113446
[1, 0, 0, 0, 1, 0]
0.23390486377965014
[0, 0, 1, 0, 1, 0]
0.35506941737276865
[0, 1, 0, 0, 1, 0]
0.2808965837885673
[0, 0, 1, 1, 0, 0]
0.41354730612785806
[1, 0, 0, 0, 0, 1]
0.18772714961801595
[0, 1, 0, 0, 1, 0]
0.2808965837885673
[0, 0, 1, 0, 1, 0]
0.35506941737276865
[1, 0, 0, 1, 0, 0]
0.29238275253473955
[1, 0, 0, 1, 0, 0]
0.292382752534

[0, 0, 1, 0, 0, 1]
0.30889170321113446
[1, 0, 0, 1, 0, 0]
0.29238275253473955
[1, 0, 0, 0, 0, 1]
0.18772714961801595
[0, 0, 1, 1, 0, 0]
0.41354730612785806
[0, 0, 1, 0, 0, 1]
0.30889170321113446
[0, 0, 1, 0, 1, 0]
0.35506941737276865
[1, 0, 0, 0, 1, 0]
0.23390486377965014
[1, 0, 0, 0, 1, 0]
0.23390486377965014
[1, 0, 0, 1, 0, 0]
0.29238275253473955
[1, 0, 0, 0, 0, 1]
0.18772714961801595
[1, 0, 0, 0, 1, 0]
0.23390486377965014
[1, 0, 0, 1, 0, 0]
0.29238275253473955
[0, 0, 1, 0, 1, 0]
0.35506941737276865
[0, 0, 1, 0, 1, 0]
0.35506941737276865
[1, 0, 0, 0, 1, 0]
0.23390486377965014
[0, 0, 1, 0, 1, 0]
0.35506941737276865
[0, 0, 1, 0, 1, 0]
0.35506941737276865
[0, 1, 0, 0, 0, 1]
0.2347188696269331
[0, 1, 0, 0, 1, 0]
0.2808965837885673
[0, 1, 0, 0, 1, 0]
0.2808965837885673
[1, 0, 0, 0, 1, 0]
0.23390486377965014
[0, 1, 0, 0, 1, 0]
0.2808965837885673
[0, 0, 1, 0, 1, 0]
0.35506941737276865
[1, 0, 0, 0, 1, 0]
0.23390486377965014
[1, 0, 0, 0, 1, 0]
0.23390486377965014
[0, 0, 1, 0, 1, 0]
0.35506941

[0, 0, 1, 0, 0, 1]
0.9882576614972418
[1, 0, 0, 0, 1, 0]
-0.1362202415715832
[1, 0, 0, 0, 1, 0]
-0.1362202415715832
[0, 0, 1, 0, 1, 0]
0.9554194917598742
[0, 0, 1, 0, 1, 0]
0.9554194917598742
[0, 1, 0, 0, 1, 0]
0.6486730698511263
[0, 0, 1, 0, 1, 0]
0.9554194917598742
[1, 0, 0, 0, 1, 0]
-0.1362202415715832
[1, 0, 0, 0, 1, 0]
-0.1362202415715832
[0, 0, 1, 0, 1, 0]
0.9554194917598742
[0, 0, 1, 1, 0, 0]
0.562038852280905
[1, 0, 0, 0, 1, 0]
-0.1362202415715832
[0, 0, 1, 0, 1, 0]
0.9554194917598742
[1, 0, 0, 0, 0, 1]
-0.10338207183421555
[0, 0, 1, 0, 1, 0]
0.9554194917598742
[0, 1, 0, 0, 1, 0]
0.6486730698511263
[1, 0, 0, 0, 1, 0]
-0.1362202415715832
[0, 0, 1, 0, 1, 0]
0.9554194917598742
[0, 1, 0, 0, 1, 0]
0.6486730698511263
[0, 1, 0, 0, 1, 0]
0.6486730698511263
[1, 0, 0, 0, 0, 1]
-0.10338207183421555
[0, 0, 1, 0, 1, 0]
0.9554194917598742
[1, 0, 0, 0, 1, 0]
-0.1362202415715832
[1, 0, 0, 0, 1, 0]
-0.1362202415715832
[0, 0, 1, 1, 0, 0]
0.562038852280905
[0, 1, 0, 0, 1, 0]
0.6486730698511263
[1

[0, 0, 1, 0, 1, 0]
0.9554194917598742
[0, 1, 0, 0, 1, 0]
0.6486730698511263
[1, 0, 0, 0, 1, 0]
-0.1362202415715832
[0, 1, 0, 0, 1, 0]
0.6486730698511263
[0, 0, 1, 0, 1, 0]
0.9554194917598742
[1, 0, 0, 1, 0, 0]
-0.5296008810505524
[1, 0, 0, 0, 0, 1]
-0.10338207183421555
[0, 0, 1, 0, 1, 0]
0.9554194917598742
[1, 0, 0, 0, 1, 0]
-0.1362202415715832
[1, 0, 0, 0, 1, 0]
-0.1362202415715832
[0, 0, 1, 0, 1, 0]
0.9554194917598742
[0, 0, 1, 0, 0, 1]
0.9882576614972418
[0, 1, 0, 0, 1, 0]
0.6486730698511263
[0, 1, 0, 1, 0, 0]
0.2552924303721571
[0, 1, 0, 0, 1, 0]
0.6486730698511263
[0, 1, 0, 0, 1, 0]
0.6486730698511263
[0, 1, 0, 0, 1, 0]
0.6486730698511263
[0, 1, 0, 0, 1, 0]
0.6486730698511263
[1, 0, 0, 0, 1, 0]
-0.1362202415715832
[0, 1, 0, 0, 1, 0]
0.6486730698511263
[0, 1, 0, 0, 1, 0]
0.6486730698511263
[1, 0, 0, 0, 1, 0]
-0.1362202415715832
[0, 1, 0, 0, 1, 0]
0.6486730698511263
[0, 1, 0, 0, 1, 0]
0.6486730698511263
[0, 0, 1, 0, 1, 0]
0.9554194917598742
[0, 0, 1, 0, 1, 0]
0.9554194917598742
[0, 

In [394]:
# print policy matchings
for clustObj in clusterDct.values():
    print("Cluster id: " + str(clustObj.clusterId))
    print("Cluster size: " + str(clustObj.no_trajs))
    print("Policy match: " + str(clustObj.policyMatch))
    print("Cluster weight: " + str(clustObj.reward))
    print("statistics sofa: " + str(clustObj.statistics_sofa))
    print("statistics AL: " + str(clustObj.statistics_AL))
    print("=========================================")


Cluster id: 1
Cluster size: 2
Policy match: 5.263157894736842
Cluster weight: [-0.15760324  0.2037592   0.744893   -0.6656341  -0.91325422 -0.76562689]
statistics sofa: [0.2972972972972973, 0.3783783783783784, 0.32432432432432434]
statistics AL: [0.16216216216216217, 0.7027027027027027, 0.13513513513513514]
Cluster id: 2
Cluster size: 4
Policy match: 3.9473684210526314
Cluster weight: [-0.63584513 -0.72891915 -0.45445677  0.27128701  0.61237584  0.33575465]
statistics sofa: [0.4266666666666667, 0.26666666666666666, 0.30666666666666664]
statistics AL: [0.17333333333333334, 0.64, 0.18666666666666668]
Cluster id: 3
Cluster size: 13
Policy match: 8.097165991902834
Cluster weight: [ 0.87923972  0.51117923  0.13710364 -0.15397104  0.20420257  0.32204391]
statistics sofa: [0.3252032520325203, 0.3902439024390244, 0.2845528455284553]
statistics AL: [0.18292682926829268, 0.6382113821138211, 0.17886178861788618]
Cluster id: 4
Cluster size: 4
Policy match: 7.894736842105263
Cluster weight: [ 0.474

In [460]:
# cluster id = 6 log likelihoods for Q values
Q_vals = maxCl['Qcells'][5][0]
table = clusterDct[6].table

states = []
tot = len(table)
matches = 0
for i in range(len(table)):
    state = int(table.iloc[i]['discrete_states'])
    actionsE = table.iloc[i]['actions']
    states.append(state)
    #actionsL = table.iloc[i]['piL']
    
    # find the log likelihoods Q(s,a | R)
#     Q_s_a = Q_vals[state]

#     # get top 3 log likelihoods
#     log_likelihoods_and_actions = sorted( [(x,i) for (i,x) in enumerate(Q_s_a)], reverse=True )[:3]
    
#     log_likelihood_E = Q_s_a[int(actionsE)]
    
#     print(state, log_likelihood_E, actionsE)
#     print(log_likelihoods_and_actions)
    #for Q_val, action in log_likelihoods_and_actions:
        #print(Q)

states = np.unique(states)

In [461]:
states

array([  0,   1,   2,   3,   4,   5,   6,   9,  11,  30,  31,  32,  33,
        34,  35,  36,  38,  39,  40,  42,  44,  60,  61,  62,  63,  64,
        65,  66,  67,  70,  73,  75,  90,  91,  92,  93,  94,  95,  96,
        99, 102, 105, 108, 120, 121, 122, 123, 124, 125, 126, 127, 128,
       129, 130, 131, 132, 136, 143, 150, 151, 152, 153, 154, 155, 156,
       157, 158, 159, 160, 162, 164, 169, 180, 181, 182, 183, 184, 185,
       186, 187, 188, 189, 192, 193, 198, 203, 210, 211, 212, 213, 214,
       215, 216, 217, 218, 219, 220, 222, 224, 240, 241, 242, 243, 244,
       245, 246, 247, 248, 249, 270, 271, 272, 273, 274, 275, 276, 277,
       278, 279, 281, 283, 287, 300, 301, 302, 303, 304, 305, 306, 307,
       308, 309, 330, 331, 332, 333, 334, 335, 336, 337, 338, 340, 342,
       344, 360, 361, 362, 363, 364, 365, 366, 367, 369, 370, 372, 390,
       391, 392, 393, 394, 395, 396, 397, 398, 420, 421, 422, 423, 424,
       425, 426, 431, 436, 450, 451, 452, 453, 454, 455, 456, 45

In [443]:
# Compute the probability distribution of actions for each state
action_distr = {}
for i in range(len(mimic)):
    state = mimic.iloc[i]['discrete_states']
    action = mimic.iloc[i]['actions']
    
    if state not in action_distr:
        action_distr[state] = [0 for i in range(25)]
    
    action_distr[state][int(action)] += 1

# normalise action distribution values
for state in action_distr:
    tot = np.sum(action_distr[state])
    vals = [x/tot for x in action_distr[state]]
    action_distr[state] = vals

In [462]:
# top 3 action distributions for expert
state_dct = {}
for state in states:
    Q_s_a = Q_vals[state]
    
    # how many top 3 Q values match with expert top 3 action
    state_dct[state] = 0
    
    # get top 3 log likelihoods for learnt policy
    log_likelihoods_and_actions_L = sorted( [(x,i) for (i,x) in enumerate(Q_s_a)], reverse=True )[:3]
    
    # find top 3 log likelihoods for experts action at that state
    action_distrs = action_distr[state]
    
    # top 3 actions
    probs_and_actions = sorted( [(x,i) for (i,x) in enumerate(action_distrs)], reverse=True )[:3]
    for prob, action in probs_and_actions:
        # if any of top 3 action appears in top 3 likelihoods of the learnt LLH policy
        for Q_val, actionL in log_likelihoods_and_actions_L:
            if action == actionL:
                state_dct[state] += 1
                

    

[(0.27476635514018694, 0), (0.205607476635514, 15), (0.1925233644859813, 5)]
[(0.26358317079966564, 0), (0.20813597102256895, 15), (0.19420451379214265, 5)]
[(0.28365068942875904, 0), (0.20945502298095864, 15), (0.18844386080105055, 5)]
[(0.2876712328767123, 0), (0.19960861056751467, 15), (0.1917808219178082, 5)]
[(0.2529411764705882, 0), (0.24705882352941178, 5), (0.19411764705882353, 15)]
[(0.36036036036036034, 0), (0.1981981981981982, 15), (0.1891891891891892, 20)]
[(0.3333333333333333, 15), (0.27450980392156865, 5), (0.23529411764705882, 0)]
[(0.2692307692307692, 0), (0.23076923076923078, 10), (0.19230769230769232, 20)]
[(0.42857142857142855, 0), (0.2857142857142857, 5), (0.14285714285714285, 20)]
[(0.2593619972260749, 0), (0.210124826629681, 15), (0.20110957004160887, 5)]
[(0.2552706552706553, 0), (0.21158594491927826, 15), (0.2074074074074074, 5)]
[(0.25203252032520324, 0), (0.22306910569105692, 15), (0.18953252032520326, 5)]
[(0.24653739612188366, 0), (0.21606648199445982, 15), 

In [452]:
state_dct

{0: 0,
 1: 0,
 2: 0,
 3: 0,
 4: 1,
 5: 2,
 6: 2,
 9: 1,
 11: 2,
 30: 3,
 31: 2,
 32: 2,
 33: 3,
 34: 2,
 35: 2,
 36: 2,
 38: 3,
 39: 1,
 40: 3,
 42: 3,
 44: 3,
 60: 2,
 61: 1,
 62: 2,
 63: 3,
 64: 2,
 65: 2,
 66: 2,
 67: 2,
 70: 2,
 73: 2,
 75: 3,
 90: 2,
 91: 0,
 92: 0,
 93: 0,
 94: 0,
 95: 2,
 96: 1,
 99: 1,
 102: 2,
 105: 3,
 108: 3,
 120: 0,
 121: 0,
 122: 0,
 123: 0,
 124: 0,
 125: 1,
 126: 1,
 127: 1,
 128: 1,
 129: 1,
 130: 3,
 131: 1,
 132: 1,
 136: 3,
 143: 3,
 150: 0,
 151: 0,
 152: 0,
 153: 0,
 154: 0,
 155: 1,
 156: 2,
 157: 0,
 158: 3,
 159: 1,
 160: 2,
 162: 2,
 164: 2,
 169: 3,
 180: 0,
 181: 2,
 182: 0,
 183: 0,
 184: 0,
 185: 0,
 186: 0,
 187: 2,
 188: 2,
 189: 1,
 192: 1,
 193: 1,
 198: 3,
 203: 3,
 210: 0,
 211: 1,
 212: 0,
 213: 0,
 214: 0,
 215: 0,
 216: 0,
 217: 0,
 218: 0,
 219: 0,
 220: 0,
 222: 2,
 224: 2,
 240: 0,
 241: 1,
 242: 1,
 243: 0,
 244: 0,
 245: 0,
 246: 0,
 247: 0,
 248: 0,
 249: 0,
 270: 0,
 271: 1,
 272: 1,
 273: 2,
 274: 0,
 275: 0,
 276: 1,
 277

In [456]:
no_states = len(state_dct)
matches = 0
print(no_states)

227


In [457]:
for key in state_dct:
    if state_dct[key] != 0:
        matches += 1        

In [458]:
print(matches)

117
